# 03 · Evaluación, experimento de inferencia y empaquetado

1. Baseline sin adapter y evaluación de las configs A y B (greedy, sin `ejecutar`).
2. Se elige el mejor adapter y se completan los experimentos de inferencia (beam, esquema crudo, `ejecutar`+reparar).
3. Se copia el adapter ganador a `adapter/` y se generan `validacion_propia.csv`, `prediccion.json` y `presentacion.pdf`.

In [ ]:
# --- Preparar el entorno (igual que notebook 01) ---
import os
REPO = "USUARIO/proyecto3-text2sql"   # <-- cambiar por el repo de la pareja
if not os.path.exists("/content/proyecto3-text2sql"):
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
    os.system(f"git clone https://{token}@github.com/{REPO} /content/proyecto3-text2sql")
os.chdir("/content/proyecto3-text2sql")
!pip install -q -r requirements.txt | tail -1
!bash scripts/instalar_mysql_colab.sh
if not os.path.exists('datos/esquemas/train/tienda_online.sql'):
    !python datos/construir_esquemas.py --socket /tmp/mysql.sock --crear-lector
    !python datos/validar_pares.py --socket /tmp/mysql.sock
    !python datos/generar_reparaciones.py
if not os.path.exists('datos/entrenamiento/dataset_chat.jsonl'):
    !python src/datos_chat.py
!python datos/validacion/verificar_validacion.py --socket /tmp/mysql.sock | tail -3


## 1. Baseline y configs A / B (greedy, esquema compacto, sin `ejecutar`)

In [ ]:
!python src/correr_evaluacion.py --nombre base_greedy --sin-adapter --beams 1 --socket /tmp/mysql.sock

In [ ]:
!python src/correr_evaluacion.py --nombre A_greedy --adapter adapters/config_A --beams 1 --socket /tmp/mysql.sock

In [ ]:
!python src/correr_evaluacion.py --nombre B_greedy --adapter adapters/config_B --beams 1 --socket /tmp/mysql.sock


## 2. Elegir el adapter ganador

In [ ]:
import json
A = json.load(open('resultados/eval_A_greedy.json'))['EX_ponderada']
B = json.load(open('resultados/eval_B_greedy.json'))['EX_ponderada']
base = json.load(open('resultados/eval_base_greedy.json'))['EX_ponderada']
# comparación con la hipótesis (resultados/hipotesis.md)
print(f"base={base:.3f} · A={A:.3f} · B={B:.3f} · mejora del mejor adapter={max(A,B)-base:+.3f} (el PDF pide >= +0.15)")
MEJOR = 'A' if A >= B else 'B'
ADAPTER = f'adapters/config_{MEJOR}'
print('Adapter ganador: config', MEJOR, '->', ADAPTER)


## 3. Experimentos de inferencia: beam 4, esquema crudo, y `ejecutar` + reparar

In [ ]:
!python src/correr_evaluacion.py --nombre mejor_beam --adapter {ADAPTER} --beams 4 --socket /tmp/mysql.sock

In [ ]:
!python src/correr_evaluacion.py --nombre mejor_crudo --adapter {ADAPTER} --formato crudo --beams 1 --socket /tmp/mysql.sock

In [ ]:
!python src/correr_evaluacion.py --nombre base_final --sin-adapter --beams 4 --socket /tmp/mysql.sock

In [ ]:
!python src/correr_evaluacion.py --nombre final --adapter {ADAPTER} --beams 4 --socket /tmp/mysql.sock
# esta corrida escribe resultados/validacion_propia.csv (entregable del proyecto)


## 4. Resumen del experimento de inferencia

In [ ]:
import json
exp = ['base_greedy', 'A_greedy', 'B_greedy', 'mejor_beam', 'mejor_crudo', 'base_final', 'final']
print('      corrida            EX     descripción')
desc = {'mejor_crudo': 'esquema crudo (sin compactar)'}
for n in exp:
    e = json.load(open(f'resultados/eval_{n}.json'))
    print(f'{n:>18}: {e["EX_ponderada"]:.3f} · F {e["por_nivel"]["facil"]:.2f} / M {e["por_nivel"]["media"]:.2f} / D {e["por_nivel"]["dificil"]:.2f}')


## 5. Copiar el adapter ganador a `adapter/` (entregable)

In [ ]:
!rm -rf adapter/*
!cp -r {ADAPTER}/. adapter/
!ls -la adapter/


## 6. Predicción justificada del EX oculto (bootstrap sobre la validación final)

In [ ]:
!python src/prediccion.py
print(open('resultados/prediccion.json').read()[:1200])


## 7. Verificar el contrato de solución (código a entregar) y presentación

In [ ]:
# Verificación de contrato con el adapter de adapter/ (como el docente):
#  - carga < 5 min (el docente máx. 5) y < 30 s por pregunta;
#  - determinismo: correr 2 veces la misma pregunta da exactamente el mismo SQL.
import time, importlib, json
t0 = time.time()
import sys; sys.path.insert(0, '.')
import solucion
json_cfg = json.load(open('config.json'))
print('config.json:', json_cfg)
print('carga completa:', round(time.time() - t0, 1), 's')
esquema = open('datos/esquemas/val/biblioteca.schema.sql').read()
p = json.load(open('datos/validacion/validacion_propia.json'))[2]
t0 = time.time()
sql1 = solucion.generar_sql(esquema, p['pregunta'])
t1 = time.time()
sql2 = solucion.generar_sql(esquema, p['pregunta'])
print('pregunta:', p['pregunta'])
print('SQL 1:', sql1)
print('SQL 2:', sql2)
print('determinista:', sql1 == sql2, '·', round(t1 - t0, 2), 's la primera, round 2:', round(time.time() - t1, 2), 's')
assert sql1 == sql2
assert solucion.es_segura(sql1), 'el filtro de seguridad no debe rechazar la respuesta'

## 8. `historial_git.txt` (entregable) y recordatorio de commits

In [ ]:
!git config user.name "Tu nombre"     # asegurarse de commitear con TU cuenta propia
!git log --all --date=iso --stat > historial_git.txt
!head -40 historial_git.txt